# Lab D — Agentic Contract Intelligence on Medallion Architecture

## End-to-end operating model

```text
                    LAB C — MEDALLION FOUNDATION
                               │
               ┌───────────────┴────────────────┐
               │                                │
               ▼                                ▼
      GOLD ENTERPRISE VIEW            SILVER CONTRACT CLAUSES
      structured business facts       contractual evidence
               │                                │
               └───────────────┬────────────────┘
                               ▼
                      ┌─────────────────┐
                      │ SUPERVISOR      │
                      │ plan + route    │
                      └────────┬────────┘
                               │
                               ▼
                      ┌─────────────────┐
                      │ CONTRACT AGENT  │
                      │ Gold facts      │
                      │ + Silver RAG    │
                      └────────┬────────┘
                               │
                               ▼
                      ┌─────────────────┐
                      │ RISK AGENT      │
                      │ evidence-backed │
                      │ assessment      │
                      └────────┬────────┘
                               │
                               ▼
                      ┌─────────────────┐
                      │ ACTION AGENT    │
                      │ proposed        │
                      │ follow-up       │
                      └────────┬────────┘
                               │
                               ▼
                    PRE-DEPLOYMENT EVALUATION
                               │
          ┌────────────────────┼─────────────────────┐
          ▼                    ▼                     ▼
 RetrievalGroundedness  RetrievalRelevance     Correctness
                               │
                               ▼
                    RetrievalSufficiency
                               │
                               ▼
                       QUALITY GATE
                    ┌──────────┴──────────┐
                    ▼                     ▼
                  FAIL                   PASS
                    │                     │
                    ▼                     ▼
               improve app          HUMAN APPROVAL
                                          │
                                 ┌────────┴────────┐
                                 ▼                 ▼
                              REJECT            APPROVE
                                                    │
                                                    ▼
                                              ACTION QUEUE
                                                    │
                                                    ▼
                                          enterprise follow-up
```

## Objective

Implement the full governed agentic operating model on top of Lab C:

**Gold + Silver Evidence → Supervisor → Contract Agent → Risk Agent → Action Agent → Evals → Quality Gate → Human Approval → Action Queue**

## Design principles

- **Gold** answers structured enterprise questions.
- **Silver contract clauses** provide contractual evidence.
- The Contract Agent combines Gold facts and Silver evidence.
- The Risk Agent reasons only from retrieved evidence and governed data.
- The Action Agent recommends an action but cannot execute it.
- MLflow evaluates the agent before deployment readiness is granted.
- Human approval is mandatory before an action enters the queue.
- The queue creates an auditable boundary between AI recommendation and enterprise execution.

## Required pre-deployment evaluations

- `RetrievalGroundedness()`
- `RetrievalRelevance()`
- `Correctness()`
- `RetrievalSufficiency()`

The retriever is traced as an MLflow `RETRIEVER` span so the RAG judges can inspect the retrieved clauses.


## Section 1 — Validate Lab C Medallion Assets

In [0]:
# Discover the Lab C medallion objects and stop if the required Gold or Silver assets are missing.
row = spark.sql(
    "SELECT current_catalog() AS catalog_name, current_schema() AS schema_name"
).first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]

SILVER_CONTRACT_CLAUSES = f"{CATALOG}.{SCHEMA}.silver_contract_clauses"
GOLD_ENTERPRISE_CONTRACTS = f"{CATALOG}.{SCHEMA}.gold_enterprise_contracts"
GOLD_ENTERPRISE_CONTRACT_VIEW = f"{CATALOG}.{SCHEMA}.gold_enterprise_contract_view"

AGENT_APPROVALS = f"{CATALOG}.{SCHEMA}.contract_agent_approvals"
AGENT_ACTION_QUEUE = f"{CATALOG}.{SCHEMA}.contract_agent_action_queue"

required_objects = [
    SILVER_CONTRACT_CLAUSES,
    GOLD_ENTERPRISE_CONTRACTS,
]

missing_objects = [
    object_name
    for object_name in required_objects
    if not spark.catalog.tableExists(object_name)
]

if missing_objects:
    raise RuntimeError(
        "Run Lab C first. Missing objects: "
        + ", ".join(missing_objects)
    )

print("Lab C foundation detected.")
print("Silver evidence:", SILVER_CONTRACT_CLAUSES)
print("Gold facts:", GOLD_ENTERPRISE_CONTRACTS)


## Section 2 — Configure MLflow Tracing and Evaluation

In [0]:
# Import MLflow tracing and the four required GenAI evaluation judges.
import json
import re
import uuid
from datetime import datetime

import mlflow
import pandas as pd
import numpy as np

from mlflow.entities import SpanType, Document

try:
    from mlflow.genai.scorers import (
        RetrievalGroundedness,
        RetrievalRelevance,
        Correctness,
        RetrievalSufficiency,
    )
except Exception as import_error:
    raise RuntimeError(
        'This lab requires a recent MLflow GenAI package. '
        'Run: %pip install --upgrade "mlflow[databricks]>=3.14.0" '
        'and then rerun this notebook.'
    ) from import_error

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(
    "/Shared/contract-intelligence-medallion-agent"
)

print("MLflow version:", mlflow.__version__)
print("Required evaluation scorers imported.")


## Section 3 — Configure the Reasoning Model

The POC attempts Databricks `ai_query` for:
- supervisor planning,
- risk reasoning,
- action recommendation.

If model access is unavailable, deterministic fallbacks keep the end-to-end workflow runnable.


In [0]:
# Configure the Databricks model endpoint and keep deterministic fallbacks enabled for Free Edition.
LLM_ENDPOINT = "system.ai.meta-llama-3-3-70b-instruct"
USE_LLM_REASONING = True

def clean_json_text(value):
    text = (value or "").strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

@mlflow.trace(
    name="call_contract_reasoning_model",
    span_type=SpanType.LLM
)
def call_reasoning_model(prompt: str):
    if not USE_LLM_REASONING:
        return None

    try:
        endpoint_sql = LLM_ENDPOINT.replace("'", "''")
        prompt_sql = prompt.replace("'", "''")

        return spark.sql(
            f"SELECT ai_query('{endpoint_sql}', '{prompt_sql}') AS response"
        ).first()["response"]

    except Exception as model_error:
        print("Reasoning model unavailable; deterministic fallback used.")
        print(
            type(model_error).__name__,
            str(model_error)[:300]
        )
        return None


## Section 4 — Build the Gold Structured-Facts Tool

In [0]:
# Define the traced Gold tool used by the Contract Agent for governed enterprise contract facts.
from pyspark.sql import functions as F

@mlflow.trace(
    name="query_gold_contract_data",
    span_type=SpanType.TOOL
)
def query_gold_contract_data(
    intent: str,
    contract_id: str = None,
    supplier_id: str = None,
    days: int = 90,
    limit: int = 20
):
    df = spark.table(
        GOLD_ENTERPRISE_CONTRACTS
    )

    if contract_id:
        df = df.filter(
            F.col("contract_id") == contract_id
        )

    if supplier_id:
        df = df.filter(
            F.col("supplier_id") == supplier_id
        )

    if intent == "contract_detail":
        result_df = df

    elif intent == "high_risk":
        result_df = (
            df.filter(
                F.col("risk_severity") == "HIGH"
            )
            .orderBy(
                "next_milestone_date"
            )
        )

    elif intent == "payments_due":
        result_df = (
            df.filter(
                F.col("next_payment_due_date").isNotNull()
            )
            .filter(
                F.col("next_payment_due_date")
                <= F.date_add(
                    F.current_date(),
                    int(days)
                )
            )
            .orderBy(
                "next_payment_due_date"
            )
        )

    elif intent == "expiring":
        result_df = (
            df.filter(
                (F.col("days_to_expiry") >= 0)
                & (F.col("days_to_expiry") <= int(days))
            )
            .orderBy(
                "days_to_expiry"
            )
        )

    elif intent == "renewal_due":
        result_df = (
            df.filter(
                F.col("renewal_review_due") == True
            )
            .orderBy(
                "days_to_expiry"
            )
        )

    elif intent == "portfolio":
        result_df = (
            df.orderBy(
                F.col("risk_severity").desc(),
                F.col("next_milestone_date")
            )
        )

    else:
        raise ValueError(
            "Supported intents: contract_detail, high_risk, "
            "payments_due, expiring, renewal_due, portfolio"
        )

    return [
        row.asDict(
            recursive=True
        )
        for row in result_df.limit(
            int(limit)
        ).collect()
    ]


## Section 5 — Build the Silver Evidence Retriever

The retriever:
- searches `silver_contract_clauses`,
- returns the most relevant clauses,
- creates an MLflow `RETRIEVER` span,
- sets the span output to MLflow `Document` objects.

This trace schema is required for the RAG evaluation judges.


In [0]:
# Build a local TF-IDF search index over the Silver contract clause table.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

clause_pdf = (
    spark.table(
        SILVER_CONTRACT_CLAUSES
    )
    .toPandas()
    .reset_index(drop=True)
)

retrieval_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

clause_matrix = retrieval_vectorizer.fit_transform(
    clause_pdf[
        "clause_text"
    ].fillna("")
)

print(
    "Silver clause chunks indexed:",
    len(clause_pdf)
)


In [0]:
# Define the MLflow RETRIEVER span that exposes Silver contract clauses to the RAG evaluation judges.
@mlflow.trace(
    name="retrieve_silver_contract_evidence",
    span_type=SpanType.RETRIEVER
)
def retrieve_silver_contract_evidence(
    query: str,
    contract_id: str = None,
    top_k: int = 5
):
    candidate_pdf = clause_pdf.copy()

    if contract_id:
        candidate_pdf = candidate_pdf[
            candidate_pdf[
                "contract_id"
            ] == contract_id
        ].copy()

    if candidate_pdf.empty:
        active_span = mlflow.get_current_active_span()
        if active_span is not None:
            active_span.set_outputs([])
        return []

    candidate_indices = (
        candidate_pdf.index.to_list()
    )

    query_vector = (
        retrieval_vectorizer
        .transform(
            [query]
        )
    )

    relevance_scores = (
        cosine_similarity(
            query_vector,
            clause_matrix[
                candidate_indices
            ]
        )
        .flatten()
    )

    candidate_pdf[
        "relevance_score"
    ] = relevance_scores

    result_pdf = (
        candidate_pdf
        .sort_values(
            "relevance_score",
            ascending=False
        )
        .head(
            int(top_k)
        )
    )

    documents = [
        Document(
            page_content=row[
                "clause_text"
            ],
            metadata={
                "doc_uri": row[
                    "document_name"
                ],
                "chunk_id": row[
                    "chunk_id"
                ],
                "contract_id": row[
                    "contract_id"
                ],
                "clause_type": row[
                    "clause_type"
                ],
                "relevance_score": float(
                    row[
                        "relevance_score"
                    ]
                ),
            },
            id=row[
                "chunk_id"
            ],
        )
        for _, row in result_pdf.iterrows()
    ]

    active_span = (
        mlflow.get_current_active_span()
    )

    if active_span is not None:
        active_span.set_outputs(
            documents
        )

    return [
        doc.to_dict()
        for doc in documents
    ]


## Section 6 — Supervisor Agent

In [0]:
# Define the Supervisor Agent that identifies the contract scope and routes the request to the correct Gold intent.
@mlflow.trace(
    name="supervisor_agent",
    span_type=SpanType.AGENT
)
def supervisor_agent(
    question: str
):
    contract_match = re.search(
        r"CTR-\d{4}-\d{3}",
        question,
        flags=re.I
    )

    contract_id = (
        contract_match.group(0).upper()
        if contract_match
        else None
    )

    question_lower = (
        question.lower()
    )

    if contract_id:
        intent = "contract_detail"

    elif "renew" in question_lower:
        intent = "renewal_due"

    elif "payment" in question_lower:
        intent = "payments_due"

    elif "expir" in question_lower:
        intent = "expiring"

    elif (
        "risk" in question_lower
        or "attention" in question_lower
        or "intervention" in question_lower
    ):
        intent = "high_risk"

    else:
        intent = "portfolio"

    fallback_plan = {
        "contract_id": contract_id,
        "structured_intent": intent,
        "retrieve_evidence": True,
        "run_risk_agent": True,
        "run_action_agent": True,
    }

    prompt = f"""
You are the Supervisor Agent for an enterprise contract intelligence system.

User request:
{question}

Current plan:
{json.dumps(fallback_plan)}

Return ONLY valid JSON:
{{
  "contract_id": "contract id or null",
  "structured_intent": "contract_detail|high_risk|payments_due|expiring|renewal_due|portfolio",
  "retrieve_evidence": true,
  "run_risk_agent": true,
  "run_action_agent": true
}}
"""

    response = call_reasoning_model(
        prompt
    )

    if response:
        try:
            parsed_plan = json.loads(
                clean_json_text(
                    response
                )
            )

            for key in fallback_plan:
                if key in parsed_plan:
                    fallback_plan[
                        key
                    ] = parsed_plan[
                        key
                    ]

        except Exception:
            pass

    return fallback_plan


## Section 7 — Contract Agent

The Contract Agent combines:
- **Gold structured facts**
- **Silver retrieved clauses**

It creates the evidence package used by downstream reasoning.


In [0]:
# Define the Contract Agent that combines Gold enterprise facts with Silver contractual evidence.
@mlflow.trace(
    name="contract_agent",
    span_type=SpanType.AGENT
)
def contract_agent(
    question: str,
    plan: dict
):
    structured_facts = (
        query_gold_contract_data(
            intent=plan[
                "structured_intent"
            ],
            contract_id=plan.get(
                "contract_id"
            ),
            days=120,
            limit=10
        )
    )

    focus_contract_id = (
        plan.get(
            "contract_id"
        )
    )

    if (
        not focus_contract_id
        and structured_facts
    ):
        focus_contract_id = (
            structured_facts[
                0
            ].get(
                "contract_id"
            )
        )

    evidence = []

    if plan.get(
        "retrieve_evidence",
        True
    ):
        evidence = (
            retrieve_silver_contract_evidence(
                query=question,
                contract_id=focus_contract_id,
                top_k=5
            )
        )

    return {
        "focus_contract_id": (
            focus_contract_id
        ),
        "structured_facts": (
            structured_facts
        ),
        "retrieved_evidence": (
            evidence
        ),
    }


## Section 8 — Risk Agent

The Risk Agent must not invent contract terms.

Its role is to interpret:
- current risk,
- milestone exposure,
- payment dependency,
- SLA / penalty exposure,
- renewal risk,

using only Gold facts and Silver evidence.


In [0]:
# Define the Risk Agent that creates an evidence-backed assessment from Gold facts and Silver clauses.
@mlflow.trace(
    name="risk_agent",
    span_type=SpanType.AGENT
)
def risk_agent(
    question: str,
    contract_context: dict
):
    facts = contract_context.get(
        "structured_facts",
        []
    )

    evidence = contract_context.get(
        "retrieved_evidence",
        []
    )

    primary_fact = (
        facts[0]
        if facts
        else {}
    )

    evidence_text = (
        evidence[0].get(
            "page_content"
        )
        if evidence
        else "No contract clause evidence retrieved."
    )

    fallback_result = {
        "contract_id": primary_fact.get(
            "contract_id"
        ),
        "supplier_name": primary_fact.get(
            "supplier_name"
        ),
        "risk_type": primary_fact.get(
            "risk_type",
            "Unknown"
        ),
        "risk_severity": primary_fact.get(
            "risk_severity",
            "UNKNOWN"
        ),
        "risk_rationale": (
            f"Gold contract data identifies "
            f"{primary_fact.get('risk_type', 'contract exposure')} "
            f"with severity "
            f"{primary_fact.get('risk_severity', 'UNKNOWN')}."
        ),
        "evidence_summary": evidence_text,
    }

    prompt = f"""
You are the Risk Agent for an enterprise contract intelligence system.

Use ONLY the supplied Gold facts and Silver contract evidence.
Do not invent contract clauses, dates, amounts, penalties, or obligations.

USER REQUEST:
{question}

GOLD FACTS:
{json.dumps(facts, default=str)}

SILVER EVIDENCE:
{json.dumps(evidence, default=str)}

Return ONLY valid JSON:
{{
  "contract_id": "string or null",
  "supplier_name": "string or null",
  "risk_type": "string",
  "risk_severity": "LOW|MEDIUM|HIGH|UNKNOWN",
  "risk_rationale": "concise evidence-backed explanation",
  "evidence_summary": "relevant contract evidence"
}}
"""

    response = call_reasoning_model(
        prompt
    )

    if response:
        try:
            return json.loads(
                clean_json_text(
                    response
                )
            )
        except Exception:
            pass

    return fallback_result


## Section 9 — Action Agent

The Action Agent can **recommend**, but it cannot execute.

Its output becomes the proposal submitted to the human approval gate.


In [0]:
# Define the Action Agent that converts the risk assessment into a controlled follow-up proposal.
@mlflow.trace(
    name="action_agent",
    span_type=SpanType.AGENT
)
def action_agent(
    question: str,
    contract_context: dict,
    risk_assessment: dict
):
    facts = contract_context.get(
        "structured_facts",
        []
    )

    primary_fact = (
        facts[0]
        if facts
        else {}
    )

    severity = risk_assessment.get(
        "risk_severity",
        "UNKNOWN"
    )

    priority_map = {
        "HIGH": "P1",
        "MEDIUM": "P2",
        "LOW": "P3",
        "UNKNOWN": "P3",
    }

    fallback_result = {
        "contract_id": risk_assessment.get(
            "contract_id"
        ),
        "supplier_name": risk_assessment.get(
            "supplier_name"
        ),
        "action_type": "CONTRACT_FOLLOW_UP",
        "proposed_action": (
            primary_fact.get(
                "recommended_action"
            )
            or "Request contract owner review."
        ),
        "owner": primary_fact.get(
            "contract_owner",
            "Procurement"
        ),
        "priority": priority_map.get(
            severity,
            "P3"
        ),
        "rationale": risk_assessment.get(
            "risk_rationale"
        ),
        "evidence": risk_assessment.get(
            "evidence_summary"
        ),
    }

    prompt = f"""
You are the Action Agent for an enterprise contract intelligence system.

Use only the supplied Gold facts and risk assessment.
Recommend a follow-up action.
Do NOT claim the action has been executed.

USER REQUEST:
{question}

GOLD FACTS:
{json.dumps(facts, default=str)}

RISK ASSESSMENT:
{json.dumps(risk_assessment, default=str)}

Return ONLY valid JSON:
{{
  "contract_id": "string or null",
  "supplier_name": "string or null",
  "action_type": "CONTRACT_FOLLOW_UP",
  "proposed_action": "specific recommended follow-up",
  "owner": "Procurement|Finance|IT|Operations",
  "priority": "P1|P2|P3",
  "rationale": "evidence-backed reason",
  "evidence": "supporting contractual evidence"
}}
"""

    response = call_reasoning_model(
        prompt
    )

    if response:
        try:
            return json.loads(
                clean_json_text(
                    response
                )
            )
        except Exception:
            pass

    return fallback_result


## Section 10 — Assemble the Complete Agentic Cycle

In [0]:
# Assemble Supervisor, Contract, Risk, and Action Agents into one traced root agent response.
@mlflow.trace(
    name="medallion_contract_agent",
    span_type=SpanType.AGENT
)
def run_agentic_cycle(
    question: str
):
    plan = supervisor_agent(
        question
    )

    contract_context = (
        contract_agent(
            question,
            plan
        )
    )

    risk = risk_agent(
        question,
        contract_context
    )

    action = action_agent(
        question,
        contract_context,
        risk
    )

    facts = contract_context.get(
        "structured_facts",
        []
    )

    primary_fact = (
        facts[0]
        if facts
        else {}
    )

    final_response = (
        f"Contract: "
        f"{action.get('contract_id') or 'portfolio-level'}\n"
        f"Supplier: "
        f"{action.get('supplier_name') or primary_fact.get('supplier_name', 'N/A')}\n"
        f"Risk: "
        f"{risk.get('risk_type')} / {risk.get('risk_severity')}\n"
        f"Next milestone: "
        f"{primary_fact.get('next_milestone', 'N/A')}\n"
        f"Next milestone date: "
        f"{primary_fact.get('next_milestone_date', 'N/A')}\n"
        f"Next payment: "
        f"{primary_fact.get('next_payment_amount', 'N/A')} "
        f"{primary_fact.get('next_payment_currency', primary_fact.get('currency', ''))}\n"
        f"Payment due: "
        f"{primary_fact.get('next_payment_due_date', 'N/A')}\n"
        f"Delay penalty: "
        f"{primary_fact.get('delay_penalty_pct_per_week', 'N/A')}% per week\n"
        f"Renewal review due: "
        f"{primary_fact.get('renewal_review_due', 'N/A')}\n"
        f"Recommended action: "
        f"{action.get('proposed_action')}\n"
        f"Priority: "
        f"{action.get('priority')}\n"
        f"Rationale: "
        f"{action.get('rationale')}\n"
        f"Evidence: "
        f"{action.get('evidence')}"
    )

    return {
        "response": final_response,
        "plan": plan,
        "contract_context": contract_context,
        "risk_assessment": risk,
        "proposed_action": action,
    }


In [0]:
# Define the simple prediction wrapper used by MLflow batch evaluation.
def agent_for_evaluation(
    question: str
):
    return run_agentic_cycle(
        question
    )[
        "response"
    ]


## Section 11 — Development Trace Test

This produces one complete trace before the formal evaluation run.

The trace should contain:
- root agent,
- Supervisor,
- Gold tool,
- Silver retriever,
- Risk Agent,
- Action Agent,
- LLM spans where available.


In [0]:
# Run one complete development trace on a high-risk Gold contract.
sample_contract_row = (
    spark.table(
        GOLD_ENTERPRISE_CONTRACTS
    )
    .filter(
        F.col(
            "risk_severity"
        ) == "HIGH"
    )
    .orderBy(
        "contract_id"
    )
    .limit(1)
    .collect()
)

if not sample_contract_row:
    sample_contract_row = (
        spark.table(
            GOLD_ENTERPRISE_CONTRACTS
        )
        .orderBy(
            "contract_id"
        )
        .limit(1)
        .collect()
    )

SAMPLE_CONTRACT_ID = (
    sample_contract_row[
        0
    ][
        "contract_id"
    ]
)

development_question = (
    f"For contract {SAMPLE_CONTRACT_ID}, identify the current risk, "
    "next milestone, payment exposure, delay penalty, renewal exposure, "
    "and the recommended follow-up action."
)

development_result = (
    run_agentic_cycle(
        development_question
    )
)

print(
    development_result[
        "response"
    ]
)


## Section 12 — Build the Pre-Deployment Evaluation Dataset

The dataset is generated from Gold truth.

Each evaluation row provides:

```python
{
  "inputs": {"question": "..."},
  "expectations": {
      "expected_facts": [...]
  }
}
```

`expected_facts` supports both:
- `Correctness`
- `RetrievalSufficiency`


In [0]:
# Build evaluation examples from Gold contract facts for correctness and retrieval-sufficiency ground truth.
evaluation_source_rows = (
    spark.table(
        GOLD_ENTERPRISE_CONTRACTS
    )
    .filter(
        F.col(
            "risk_severity"
        ).isin(
            "HIGH",
            "MEDIUM"
        )
    )
    .orderBy(
        "contract_id"
    )
    .limit(4)
    .collect()
)

eval_dataset = []

for source_row in evaluation_source_rows:
    record = source_row.asDict(
        recursive=True
    )

    question = (
        f"For contract {record['contract_id']}, identify the supplier, "
        "risk type and severity, next milestone, next payment exposure, "
        "delay penalty, and recommended follow-up."
    )

    expected_facts = [
        f"Contract ID is {record['contract_id']}",
        f"Supplier is {record['supplier_name']}",
        f"Risk type is {record['risk_type']}",
        f"Risk severity is {record['risk_severity']}",
        f"Next milestone is {record['next_milestone']}",
        (
            f"Next payment amount is "
            f"{record['next_payment_amount']} "
            f"{record['next_payment_currency']}"
        ),
        (
            f"Delay penalty is "
            f"{record['delay_penalty_pct_per_week']} "
            f"percent per week"
        ),
        (
            f"Recommended action is "
            f"{record['recommended_action']}"
        ),
    ]

    eval_dataset.append(
        {
            "inputs": {
                "question": question
            },
            "expectations": {
                "expected_facts": (
                    expected_facts
                )
            },
        }
    )

print(
    "Evaluation examples:",
    len(eval_dataset)
)

print(
    json.dumps(
        eval_dataset[0],
        indent=2,
        default=str
    )
)


## Section 13 — Run the Four Required MLflow Evals

The built-in Databricks judge model is used by default.

### 1. RetrievalGroundedness
Is the response supported by retrieved Silver evidence?

### 2. RetrievalRelevance
Are the retrieved clauses relevant to the user's request?

### 3. Correctness
Does the final response contain the expected Gold facts?

### 4. RetrievalSufficiency
Did retrieval provide enough evidence to support the expected facts?


In [0]:
# Run the four required MLflow GenAI judges against traces generated by the agent.
EVALUATION_COMPLETED = False
evaluation_results = None

try:
    evaluation_results = (
        mlflow.genai.evaluate(
            data=eval_dataset,
            predict_fn=agent_for_evaluation,
            scorers=[
                RetrievalGroundedness(),
                RetrievalRelevance(),
                Correctness(),
                RetrievalSufficiency(),
            ],
        )
    )

    EVALUATION_COMPLETED = True

    print(
        "Evaluation run ID:",
        evaluation_results.run_id
    )

    print(
        "Aggregated metrics:"
    )

    print(
        json.dumps(
            evaluation_results.metrics,
            indent=2,
            default=str
        )
    )

    display(
        evaluation_results.result_df
    )

except Exception as evaluation_error:
    print(
        "MLflow GenAI evaluation could not complete."
    )

    print(
        type(
            evaluation_error
        ).__name__,
        str(
            evaluation_error
        )[:1200]
    )


## Section 14 — Pre-Deployment Quality Gate

The application is not marked deployment-ready merely because the code runs.

For this POC:

- all four evaluation families must be returned,
- each score must meet the threshold.

The example threshold is **0.75**.

A production implementation should set thresholds based on business risk and validated evaluator performance.


In [0]:
# Convert the four evaluation results into an explicit pre-deployment quality gate.
MIN_EVAL_SCORE = 0.75

def find_evaluation_metric(
    metrics,
    family
):
    preferred_keys = [
        f"{family}/mean",
        f"{family}/pass_rate",
    ]

    for metric_key in preferred_keys:
        if metric_key in metrics:
            return (
                metric_key,
                float(
                    metrics[
                        metric_key
                    ]
                )
            )

    for metric_key, metric_value in metrics.items():
        if family in metric_key:
            try:
                return (
                    metric_key,
                    float(
                        metric_value
                    )
                )
            except Exception:
                pass

    return (
        None,
        None
    )

required_families = [
    "retrieval_groundedness",
    "retrieval_relevance",
    "correctness",
    "retrieval_sufficiency",
]

gate_rows = []

if EVALUATION_COMPLETED:
    for family in required_families:
        metric_key, metric_score = (
            find_evaluation_metric(
                evaluation_results.metrics,
                family
            )
        )

        gate_rows.append(
            {
                "evaluation": family,
                "metric_key": metric_key,
                "score": metric_score,
                "threshold": MIN_EVAL_SCORE,
                "passed": (
                    metric_score is not None
                    and metric_score >= MIN_EVAL_SCORE
                ),
            }
        )

else:
    for family in required_families:
        gate_rows.append(
            {
                "evaluation": family,
                "metric_key": None,
                "score": None,
                "threshold": MIN_EVAL_SCORE,
                "passed": False,
            }
        )

evaluation_gate_pdf = (
    pd.DataFrame(
        gate_rows
    )
)

display(
    evaluation_gate_pdf
)

DEPLOYMENT_READY = bool(
    EVALUATION_COMPLETED
    and evaluation_gate_pdf[
        "passed"
    ].all()
)

print(
    "DEPLOYMENT_READY =",
    DEPLOYMENT_READY
)


## Section 15 — Development-Only Override

If Free Edition cannot access an evaluator model, the operational workflow can still be demonstrated manually.

Keep this **False** for the normal governed path.

Setting it to `True` is a POC-only demonstration override and does not change `DEPLOYMENT_READY`.


In [0]:
# Keep the quality gate strict while providing an explicit POC-only override for downstream workflow demonstration.
DEVELOPMENT_DEMO_OVERRIDE = False

CAN_RUN_OPERATIONAL_DEMO = (
    DEPLOYMENT_READY
    or DEVELOPMENT_DEMO_OVERRIDE
)

print(
    "Deployment ready:",
    DEPLOYMENT_READY
)

print(
    "Development override:",
    DEVELOPMENT_DEMO_OVERRIDE
)

print(
    "Operational demo enabled:",
    CAN_RUN_OPERATIONAL_DEMO
)


## Section 16 — Create Human Approval and Action Queue Tables

In [0]:
# Create the persistent human-approval and action-queue Delta tables used after the evaluation gate.
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {AGENT_APPROVALS} (
        approval_id STRING,
        action_id STRING,
        contract_id STRING,
        supplier_name STRING,
        proposed_action STRING,
        action_owner STRING,
        priority STRING,
        rationale STRING,
        evidence STRING,
        requested_at TIMESTAMP,
        approval_status STRING,
        approver STRING,
        reviewed_at TIMESTAMP
    )
    USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {AGENT_ACTION_QUEUE} (
        action_id STRING,
        approval_id STRING,
        contract_id STRING,
        supplier_name STRING,
        action_type STRING,
        action_owner STRING,
        proposed_action STRING,
        priority STRING,
        rationale STRING,
        evidence STRING,
        queue_status STRING,
        queued_at TIMESTAMP,
        completed_at TIMESTAMP
    )
    USING DELTA
    """
)

print(
    "Approval table:",
    AGENT_APPROVALS
)

print(
    "Action queue:",
    AGENT_ACTION_QUEUE
)


## Section 17 — Submit the Agent Proposal for Human Approval

The agent cannot write directly to the action queue.

It can only create a:

`PENDING`

approval request.


In [0]:
# Define the function that converts an Action Agent proposal into a pending human-approval record.
def create_approval_request(
    agent_result: dict
):
    action = agent_result[
        "proposed_action"
    ]

    approval_id = str(
        uuid.uuid4()
    )

    action_id = str(
        uuid.uuid4()
    )

    approval_record = [
        (
            approval_id,
            action_id,
            action.get(
                "contract_id"
            ),
            action.get(
                "supplier_name"
            ),
            action.get(
                "proposed_action"
            ),
            action.get(
                "owner"
            ),
            action.get(
                "priority"
            ),
            action.get(
                "rationale"
            ),
            action.get(
                "evidence"
            ),
            datetime.utcnow(),
            "PENDING",
            None,
            None,
        )
    ]

    approval_df = (
        spark.createDataFrame(
            approval_record,
            """
            approval_id string,
            action_id string,
            contract_id string,
            supplier_name string,
            proposed_action string,
            action_owner string,
            priority string,
            rationale string,
            evidence string,
            requested_at timestamp,
            approval_status string,
            approver string,
            reviewed_at timestamp
            """
        )
    )

    approval_df.write.mode(
        "append"
    ).format(
        "delta"
    ).saveAsTable(
        AGENT_APPROVALS
    )

    return approval_id


## Section 18 — Human Approval Gate

Valid decisions:

- `APPROVE`
- `REJECT`

Only an approved proposal can be inserted into the action queue.


In [0]:
# Define the human-review function that enqueues only explicitly approved proposals.
def review_and_enqueue(
    approval_id: str,
    decision: str,
    approver: str = "POC Human Reviewer"
):
    decision = (
        decision
        .upper()
        .strip()
    )

    if decision not in {
        "APPROVE",
        "REJECT"
    }:
        raise ValueError(
            "decision must be APPROVE or REJECT"
        )

    approval_status = (
        "APPROVED"
        if decision == "APPROVE"
        else "REJECTED"
    )

    safe_approver = (
        approver.replace(
            "'",
            "''"
        )
    )

    spark.sql(
        f"""
        UPDATE {AGENT_APPROVALS}
        SET
            approval_status = '{approval_status}',
            approver = '{safe_approver}',
            reviewed_at = current_timestamp()
        WHERE approval_id = '{approval_id}'
          AND approval_status = 'PENDING'
        """
    )

    reviewed_rows = (
        spark.table(
            AGENT_APPROVALS
        )
        .filter(
            F.col(
                "approval_id"
            ) == approval_id
        )
        .limit(1)
        .collect()
    )

    if not reviewed_rows:
        raise RuntimeError(
            "Approval record not found."
        )

    record = (
        reviewed_rows[
            0
        ].asDict(
            recursive=True
        )
    )

    if record[
        "approval_status"
    ] != "APPROVED":
        return {
            "approval_status": record[
                "approval_status"
            ],
            "queued": False,
        }

    already_queued = (
        spark.table(
            AGENT_ACTION_QUEUE
        )
        .filter(
            F.col(
                "action_id"
            ) == record[
                "action_id"
            ]
        )
        .count()
    )

    if already_queued == 0:
        queue_df = spark.createDataFrame(
            [
                (
                    record[
                        "action_id"
                    ],
                    record[
                        "approval_id"
                    ],
                    record[
                        "contract_id"
                    ],
                    record[
                        "supplier_name"
                    ],
                    "CONTRACT_FOLLOW_UP",
                    record[
                        "action_owner"
                    ],
                    record[
                        "proposed_action"
                    ],
                    record[
                        "priority"
                    ],
                    record[
                        "rationale"
                    ],
                    record[
                        "evidence"
                    ],
                    "QUEUED",
                    datetime.utcnow(),
                    None,
                )
            ],
            """
            action_id string,
            approval_id string,
            contract_id string,
            supplier_name string,
            action_type string,
            action_owner string,
            proposed_action string,
            priority string,
            rationale string,
            evidence string,
            queue_status string,
            queued_at timestamp,
            completed_at timestamp
            """
        )

        queue_df.write.mode(
            "append"
        ).format(
            "delta"
        ).saveAsTable(
            AGENT_ACTION_QUEUE
        )

    return {
        "approval_status": "APPROVED",
        "queued": True,
        "action_id": record[
            "action_id"
        ],
    }


## Section 19 — Run the Governed Operational Cycle

Normal execution:

```text
agent result
   ↓
eval gate
   ↓
approval request
   ↓
human reviews
   ↓
approve / reject
   ↓
action queue
```


In [0]:
# Run the operational agent only after the evaluation gate or an explicit development-only override.
operational_question = (
    f"For contract {SAMPLE_CONTRACT_ID}, identify the current risk, "
    "milestone and payment exposure, relevant contractual evidence, "
    "and propose the appropriate follow-up."
)

operational_result = None
approval_id = None

if CAN_RUN_OPERATIONAL_DEMO:
    operational_result = (
        run_agentic_cycle(
            operational_question
        )
    )

    print(
        operational_result[
            "response"
        ]
    )

    approval_id = (
        create_approval_request(
            operational_result
        )
    )

    print(
        "Approval request created:",
        approval_id
    )

else:
    print(
        "Operational workflow stopped because "
        "the pre-deployment quality gate has not passed."
    )


In [0]:
# Simulate the human reviewer and move an approved proposal into the action queue.
HUMAN_DECISION = "APPROVE"

if approval_id is not None:
    human_review_result = (
        review_and_enqueue(
            approval_id=approval_id,
            decision=HUMAN_DECISION,
            approver="POC Human Reviewer"
        )
    )

    print(
        human_review_result
    )

else:
    print(
        "No approval request exists because "
        "the operational workflow was not enabled."
    )


## Section 20 — Inspect Governance and Action State

In [0]:
# Display the complete human-approval audit trail and approved action queue.
print(
    "HUMAN APPROVAL AUDIT"
)

display(
    spark.table(
        AGENT_APPROVALS
    )
    .orderBy(
        F.col(
            "requested_at"
        ).desc()
    )
)

print(
    "APPROVED ACTION QUEUE"
)

display(
    spark.table(
        AGENT_ACTION_QUEUE
    )
    .orderBy(
        F.col(
            "queued_at"
        ).desc()
    )
)


## Section 21 — Responsibilities by Layer and Agent

| Component | Primary responsibility |
|---|---|
| **Gold** | Governed structured enterprise contract facts |
| **Silver clauses** | Contractual evidence for retrieval |
| **Supervisor Agent** | Understand request and route work |
| **Contract Agent** | Combine Gold facts + Silver evidence |
| **Risk Agent** | Interpret contractual exposure |
| **Action Agent** | Recommend follow-up |
| **MLflow Evals** | Test retrieval and answer quality |
| **Quality Gate** | Decide deployment readiness |
| **Human Approver** | Authorize or reject proposed action |
| **Action Queue** | Persist approved enterprise follow-up |

### Key control boundary

```text
AI recommendation
      │
      ▼
quality evaluation
      │
      ▼
human authorization
      │
      ▼
enterprise execution
```

The model never autonomously crosses this boundary.


## Section 22 — Evaluation Interpretation

### RetrievalRelevance
**Question:** Did the Contract Agent retrieve clauses relevant to the request?

Useful for diagnosing:
- weak retrieval queries,
- irrelevant chunks,
- poor filtering.

### RetrievalGroundedness
**Question:** Is the final answer supported by retrieved evidence?

Useful for diagnosing:
- hallucinated clauses,
- unsupported penalties,
- invented contract terms.

### RetrievalSufficiency
**Question:** Does retrieval contain enough evidence to support the expected facts?

Useful for diagnosing:
- incomplete evidence,
- insufficient `top_k`,
- missing clause types.

### Correctness
**Question:** Does the response match known Gold truth?

Useful for diagnosing:
- wrong supplier,
- wrong payment,
- wrong milestone,
- wrong risk,
- wrong penalty.

### Combined interpretation

```text
Relevance low
    → retrieval is noisy

Relevance high + Sufficiency low
    → relevant evidence was found, but not enough

Sufficiency high + Groundedness low
    → evidence was available but the agent did not use it correctly

Groundedness high + Correctness low
    → answer may be grounded in retrieved context but conflicts with known truth
```


## Section 23 — Complete Enterprise Architecture

```text
SOURCES
  │
  ▼
BRONZE
raw suppliers • contracts • documents • Ariba • ERP • SharePoint
  │
  ▼
AI EXTRACTION
  │
  ▼
SILVER
obligations • milestones • payments • risks • clauses
  │
  ├───────────────────────┐
  │                       │
  ▼                       ▼
GOLD                 RAG EVIDENCE
enterprise view      Silver clauses
  │                       │
  └───────────┬───────────┘
              ▼
        SUPERVISOR
              │
              ▼
        CONTRACT AGENT
              │
              ▼
          RISK AGENT
              │
              ▼
         ACTION AGENT
              │
              ▼
     PRE-DEPLOYMENT EVALS
              │
              ▼
        QUALITY GATE
              │
              ▼
       HUMAN APPROVAL
              │
              ▼
        ACTION QUEUE
              │
              ▼
     enterprise follow-up
              │
              ▼
       trace + feedback
              │
              └──────► future evaluation cases
```


In [0]:
# Print the final Lab D architecture and governance status for verification.
lab_d_summary = {
    "medallion_inputs": {
        "gold_facts": (
            GOLD_ENTERPRISE_CONTRACTS
        ),
        "silver_evidence": (
            SILVER_CONTRACT_CLAUSES
        ),
    },
    "agent_flow": [
        "Supervisor Agent",
        "Contract Agent",
        "Risk Agent",
        "Action Agent",
    ],
    "pre_deployment_evals": [
        "RetrievalGroundedness",
        "RetrievalRelevance",
        "Correctness",
        "RetrievalSufficiency",
    ],
    "governance": [
        "Quality Gate",
        "Human Approval",
        "Action Queue",
    ],
    "approval_table": (
        AGENT_APPROVALS
    ),
    "action_queue_table": (
        AGENT_ACTION_QUEUE
    ),
    "evaluation_completed": (
        EVALUATION_COMPLETED
    ),
    "deployment_ready": (
        DEPLOYMENT_READY
    ),
    "development_override": (
        DEVELOPMENT_DEMO_OVERRIDE
    ),
}

print(
    json.dumps(
        lab_d_summary,
        indent=2
    )
)


## Lab D Completion Checklist

### Medallion consumption
- Gold enterprise facts used by structured tool
- Silver contract clauses used by retriever
- No duplication of Lab C ingestion or transformation logic

### Agentic cycle
- Supervisor Agent
- Contract Agent
- Risk Agent
- Action Agent

### MLflow tracing
- Root Agent span
- Agent spans
- Gold `TOOL` span
- Silver `RETRIEVER` span
- LLM spans

### Pre-deployment evaluation
- `RetrievalGroundedness()`
- `RetrievalRelevance()`
- `Correctness()`
- `RetrievalSufficiency()`
- Explicit deployment-quality gate

### Human governance
- Pending approval record
- Explicit APPROVE / REJECT decision
- Only approved proposals enter action queue
- Approval and action history persisted in Delta

## Complete POC

### Lab C
**Sources → Bronze → AI extraction → Silver → Gold → Materialized View**

### Lab D
**Gold + Silver Evidence → Supervisor → Contract Agent → Risk Agent → Action Agent → Evals → Quality Gate → Human Approval → Action Queue**

Together they implement the executive architecture end to end.
